# Módulo 1 · Datos y Optimización de Markowitz

Proyecto: Análisis y Diseño de Algoritmos · Optimización de Portafolios

Este notebook descarga datos históricos reales de Yahoo Finance y calcula los portafolios
de **Máximo Sharpe** y **Mínima Varianza** mediante el modelo de media-varianza de Markowitz
(`scipy.optimize`, método SLSQP), además de simular la evolución de riqueza (backtesting).

In [1]:
!pip install yfinance plotly -q

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from scipy.optimize import minimize
import json
import yfinance as yf

## Parámetros globales

Equivalentes a los inputs que en la app de Streamlit se recogían en el `st.sidebar`.

In [2]:
# ==============================================================
# VARIABLES GLOBALES
# ==============================================================
TICKERS = ['FSM', 'VOLCABC1.LM', 'ABX.TO', 'BVN', 'BHP']
START_DATE = '2015-01-01'
END_DATE = '2024-12-31'
CAPITAL = 100000
FREQUENCY = 'Mensual'  # 'Semanal', 'Mensual' o 'Trimestral'

## Descarga y preparación de datos

Se descargan precios ajustados desde Yahoo Finance y se calculan retornos logarítmicos,
el vector de retornos esperados anualizado ($\mu$) y la matriz de covarianza anualizada ($\Sigma$).

In [3]:
def cargar_datos(tickers, start, end):
    raw = yf.download(tickers, start=start, end=end, auto_adjust=True, progress=False)
    if isinstance(raw.columns, pd.MultiIndex):
        precios = raw['Close']
    else:
        precios = raw[['Close']]
        precios.columns = tickers
    precios = precios.dropna(how='all').ffill().dropna()
    tick_list = list(precios.columns)
    log_returns = np.log(precios / precios.shift(1)).dropna()
    mu = log_returns.mean() * 252
    cov = log_returns.cov() * 252
    return precios, log_returns, mu, cov, tick_list

precios, log_returns, mu, cov, TICK_LIST = cargar_datos(TICKERS, START_DATE, END_DATE)
retornos_simples = precios.pct_change().dropna()

print(f"Activos cargados: {TICK_LIST}")
print(f"Sesiones: {len(precios)}")
mu

Activos cargados: ['ABX.TO', 'BHP', 'BVN', 'FSM', 'VOLCABC1.LM']
Sesiones: 2560


,0
Ticker,
ABX.TO,0.071694
BHP,0.080794
BVN,0.024776
FSM,-0.010422
VOLCABC1.LM,-0.114717


## Funciones del modelo de Markowitz (media-varianza)

Se define el rendimiento esperado del portafolio, su volatilidad, y los problemas de
optimización de Máximo Sharpe y Mínima Varianza con restricciones de suma de pesos = 1
y pesos no negativos (sin apalancamiento ni ventas en corto).

In [4]:
def rendimiento_portafolio(pesos, mu, cov):
    ret = float(np.dot(pesos, mu))
    vol = float(np.sqrt(pesos @ cov @ pesos))
    return ret, vol

def _neg_sharpe(pesos, mu, cov, rf=0.0):
    ret, vol = rendimiento_portafolio(pesos, mu, cov)
    return -(ret - rf) / vol if vol > 0 else 0.0

def _volatilidad(pesos, mu, cov):
    return rendimiento_portafolio(pesos, mu, cov)[1]

def optimizar_max_sharpe(mu, cov, rf=0.0):
    n = len(mu)
    bounds = tuple((0.0, 1.0) for _ in range(n))
    cons = ({'type': 'eq', 'fun': lambda w: np.sum(w) - 1},)
    init = np.repeat(1.0 / n, n)
    res = minimize(_neg_sharpe, init, args=(mu.values, cov.values, rf),
                    method='SLSQP', bounds=bounds, constraints=cons)
    return res.x if res.success else init

def optimizar_min_varianza(mu, cov):
    n = len(mu)
    bounds = tuple((0.0, 1.0) for _ in range(n))
    cons = ({'type': 'eq', 'fun': lambda w: np.sum(w) - 1},)
    init = np.repeat(1.0 / n, n)
    res = minimize(_volatilidad, init, args=(mu.values, cov.values),
                    method='SLSQP', bounds=bounds, constraints=cons)
    return res.x if res.success else init

## Portafolios óptimos (Máx. Sharpe / Mín. Varianza)

In [5]:
w_sharpe = optimizar_max_sharpe(mu, cov)
w_minvar = optimizar_min_varianza(mu, cov)

ret_s, vol_s = rendimiento_portafolio(w_sharpe, mu.values, cov.values)
ret_mv, vol_mv = rendimiento_portafolio(w_minvar, mu.values, cov.values)
sharpe_s = ret_s / vol_s if vol_s > 0 else np.nan

print("=== Portafolio de Maximo Sharpe ===")
for t, w in zip(TICK_LIST, w_sharpe):
    print(f"  {t}: {w*100:.2f}%")
print(f"  Retorno esperado: {ret_s*100:.2f}%  |  Volatilidad: {vol_s*100:.2f}%  |  Sharpe: {sharpe_s:.2f}")

print("\n=== Portafolio de Minima Varianza ===")
for t, w in zip(TICK_LIST, w_minvar):
    print(f"  {t}: {w*100:.2f}%")
print(f"  Retorno esperado: {ret_mv*100:.2f}%  |  Volatilidad: {vol_mv*100:.2f}%")

=== Portafolio de Maximo Sharpe ===
  ABX.TO: 37.73%
  BHP: 62.27%
  BVN: 0.00%
  FSM: 0.00%
  VOLCABC1.LM: 0.00%
  Retorno esperado: 7.74%  |  Volatilidad: 27.83%  |  Sharpe: 0.28

=== Portafolio de Minima Varianza ===
  ABX.TO: 37.64%
  BHP: 48.39%
  BVN: 0.00%
  FSM: 0.00%
  VOLCABC1.LM: 13.97%
  Retorno esperado: 5.01%  |  Volatilidad: 26.72%


## Frontera eficiente

Se resuelve el problema de mínima varianza para 200 niveles de retorno objetivo,
trazando la frontera eficiente completa junto a los activos individuales.

In [6]:
def frontera_eficiente(mu, cov, n_puntos=200):
    n = len(mu)
    objetivos = np.linspace(mu.min(), mu.max(), n_puntos)
    vols, pesos_list = [], []
    bounds = tuple((0.0, 1.0) for _ in range(n))
    init = np.repeat(1.0 / n, n)
    for target in objetivos:
        cons = (
            {'type': 'eq', 'fun': lambda w: np.sum(w) - 1},
            {'type': 'eq', 'fun': lambda w, t=target: float(np.dot(w, mu.values)) - t},
        )
        res = minimize(_volatilidad, init, args=(mu.values, cov.values),
                        method='SLSQP', bounds=bounds, constraints=cons)
        if res.success:
            vols.append(res.fun); pesos_list.append(res.x)
        else:
            vols.append(np.nan); pesos_list.append(np.full(n, np.nan))
    return objetivos, np.array(vols), pesos_list

objetivos, vols_frontera, _ = frontera_eficiente(mu, cov, n_puntos=200)

fig_ef = px.line(x=vols_frontera, y=objetivos,
                  labels={'x': 'Volatilidad (Riesgo)', 'y': 'Retorno Esperado'},
                  title='Frontera Eficiente de Markowitz')
fig_ef.add_scatter(x=[vol_s], y=[ret_s], mode='markers', name='Maximo Sharpe',
                    marker=dict(size=13, color='#C4622D'))
fig_ef.add_scatter(x=[vol_mv], y=[ret_mv], mode='markers', name='Minima Varianza',
                    marker=dict(size=13, color='#3D4F4A'))
vol_activos = np.sqrt(np.diag(cov.values))
fig_ef.add_scatter(x=vol_activos, y=mu.values, mode='markers+text', name='Activos',
                    text=TICK_LIST, textposition='top center', marker=dict(size=8, color='#7C9473'))
fig_ef.show()

## Composición de los portafolios óptimos

In [7]:
fig_pie_sharpe = px.pie(names=TICK_LIST, values=w_sharpe, title='Asignacion - Maximo Sharpe',
                         color_discrete_sequence=px.colors.sequential.YlGnBu)
fig_pie_sharpe.show()

fig_pie_minvar = px.pie(names=TICK_LIST, values=w_minvar, title='Asignacion - Minima Varianza',
                         color_discrete_sequence=px.colors.sequential.Sunset)
fig_pie_minvar.show()

## Simulación de riqueza (backtesting) y persistencia de resultados

Se comparan tres estrategias con capital inicial `CAPITAL`: Buy & Hold (sin rebalanceo),
rebalanceo periódico al portafolio de Máximo Sharpe, y equiponderado. Al final se guardan
los resultados clave en `resultados_m1.json` para que el notebook de comparación (Módulo 4)
los pueda leer.

In [8]:
def _fechas_rebalanceo(precios_index, freq_label):
    freq_map = {'Semanal': 'W', 'Mensual': 'M', 'Trimestral': 'Q'}
    freq = freq_map.get(freq_label, 'M')
    serie = pd.Series(precios_index, index=precios_index)
    ultimas = serie.resample(freq).last().dropna()
    return set(ultimas.values)

def simular_riqueza(retornos_simples, pesos_objetivo, capital, freq_label=None):
    pesos_objetivo = np.array(pesos_objetivo, dtype=float)
    fechas = retornos_simples.index
    rebal_dates = _fechas_rebalanceo(fechas, freq_label) if freq_label else set()
    valores_activos = capital * pesos_objetivo
    wealth = []
    for fecha, fila in retornos_simples.iterrows():
        valores_activos = valores_activos * (1 + fila.values)
        total = valores_activos.sum()
        wealth.append(total)
        if freq_label and fecha in rebal_dates:
            valores_activos = total * pesos_objetivo
    return pd.Series(wealth, index=fechas)

w_eq = np.repeat(1.0 / len(TICK_LIST), len(TICK_LIST))
wealth_bh = simular_riqueza(retornos_simples, w_sharpe, CAPITAL, freq_label=None)
wealth_mkw = simular_riqueza(retornos_simples, w_sharpe, CAPITAL, freq_label=FREQUENCY)
wealth_eq = simular_riqueza(retornos_simples, w_eq, CAPITAL, freq_label=FREQUENCY)

fig_wealth = go.Figure()
fig_wealth.add_trace(go.Scatter(x=wealth_bh.index, y=wealth_bh.values,
                                 name='Buy & Hold (Max. Sharpe)', line=dict(color='#B3452F', dash='dash')))
fig_wealth.add_trace(go.Scatter(x=wealth_mkw.index, y=wealth_mkw.values,
                                 name=f'Rebalanceado ({FREQUENCY})', line=dict(color='#7C9473', width=3)))
fig_wealth.add_trace(go.Scatter(x=wealth_eq.index, y=wealth_eq.values,
                                 name='Equiponderado', line=dict(color='#3D4F4A', dash='dot')))
fig_wealth.update_layout(title='Evolucion de la Riqueza (Backtesting)',
                          xaxis_title='Fecha', yaxis_title='Valor del Portafolio (USD)')
fig_wealth.show()

/tmp/ipykernel_489/3244575438.py:5: FutureWarning:

'M' is deprecated and will be removed in a future version, please use 'ME' instead.



In [9]:
resultados_m1 = {
    'tickers': TICK_LIST,
    'w_sharpe': w_sharpe.tolist(),
    'w_minvar': w_minvar.tolist(),
    'ret_sharpe': ret_s, 'vol_sharpe': vol_s, 'sharpe_ratio': sharpe_s,
    'ret_minvar': ret_mv, 'vol_minvar': vol_mv,
    'frontera_objetivos': objetivos.tolist(),
    'frontera_vols': vols_frontera.tolist(),
    'wealth_bh': {'fechas': [d.strftime('%Y-%m-%d') for d in wealth_bh.index], 'valores': wealth_bh.values.tolist()},
    'wealth_mkw': {'fechas': [d.strftime('%Y-%m-%d') for d in wealth_mkw.index], 'valores': wealth_mkw.values.tolist()},
    'wealth_eq': {'fechas': [d.strftime('%Y-%m-%d') for d in wealth_eq.index], 'valores': wealth_eq.values.tolist()},
    'capital_inicial': CAPITAL,
    'frequency': FREQUENCY,
}

with open('resultados_m1.json', 'w') as f:
    json.dump(resultados_m1, f, indent=2)

print("Resultados del Modulo 1 guardados en resultados_m1.json")

Resultados del Modulo 1 guardados en resultados_m1.json
